In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn import preprocessing
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score



In [2]:

folder_name ="/Users/chandra/Desktop/FSDS_GenAI_Training/FSDS_Classes/Python_Workspace/ML/RegularizationTechniques/"
file_name ="car-mpg.csv"
full_file_path = folder_name + file_name
print("Full file path = ",full_file_path)



Full file path =  /Users/chandra/Desktop/FSDS_GenAI_Training/FSDS_Classes/Python_Workspace/ML/RegularizationTechniques/car-mpg.csv


In [3]:
# Load the dataset
cars_df = pd.read_csv(full_file_path)
cars_df.head()

,mpg,cyl,disp,hp,wt,acc,yr,origin,car_type,car_name
0,18.0,8,307.0,130,3504,12.0,70,1,0,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,0,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,0,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,0,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,0,ford torino


In [4]:
cars_df.columns

Index(['mpg', 'cyl', 'disp', 'hp', 'wt', 'acc', 'yr', 'origin', 'car_type',
       'car_name'],
      dtype='object')

# Drop car name
# Replace origin into 1,2,3.. dont forget get_dummies
# Replace ? with nan
# Replace all nan with median


In [5]:
#Drop car name
cars_df = cars_df.drop(['car_name'], axis = 1)

In [6]:
#Replace origin into 1,2,3.. dont forget get_dummies
cars_df['origin'] = cars_df['origin'].replace({1: 'america', 2: 'europe', 3: 'asia'})
# cars_df = pd.get_dummies(cars_df,columns = ['origin'])
cars_df = pd.get_dummies(cars_df,columns = ['origin'], dtype=int)

cars_df.tail(10)


,mpg,cyl,disp,hp,wt,acc,yr,car_type,origin_america,origin_asia,origin_europe
388,26.0,4,156.0,92,2585,14.5,82,1,1,0,0
389,22.0,6,232.0,112,2835,14.7,82,0,1,0,0
390,32.0,4,144.0,96,2665,13.9,82,1,0,1,0
391,36.0,4,135.0,84,2370,13.0,82,1,1,0,0
392,27.0,4,151.0,90,2950,17.3,82,1,1,0,0
393,27.0,4,140.0,86,2790,15.6,82,1,1,0,0
394,44.0,4,97.0,52,2130,24.6,82,1,0,0,1
395,32.0,4,135.0,84,2295,11.6,82,1,1,0,0
396,28.0,4,120.0,79,2625,18.6,82,1,1,0,0
397,31.0,4,119.0,82,2720,19.4,82,1,1,0,0


In [7]:
#Replace ? with nan


cars_df = cars_df.replace('?', np.nan)
cars_df.tail()

# cars_df.isna().sum()

# cars_df['hp'].isna().sum()


,mpg,cyl,disp,hp,wt,acc,yr,car_type,origin_america,origin_asia,origin_europe
393,27.0,4,140.0,86,2790,15.6,82,1,1,0,0
394,44.0,4,97.0,52,2130,24.6,82,1,0,0,1
395,32.0,4,135.0,84,2295,11.6,82,1,1,0,0
396,28.0,4,120.0,79,2625,18.6,82,1,1,0,0
397,31.0,4,119.0,82,2720,19.4,82,1,1,0,0


In [8]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   mpg             398 non-null    float64
 1   cyl             398 non-null    int64  
 2   disp            398 non-null    float64
 3   hp              392 non-null    object 
 4   wt              398 non-null    int64  
 5   acc             398 non-null    float64
 6   yr              398 non-null    int64  
 7   car_type        398 non-null    int64  
 8   origin_america  398 non-null    int64  
 9   origin_asia     398 non-null    int64  
 10  origin_europe   398 non-null    int64  
dtypes: float64(3), int64(7), object(1)
memory usage: 34.3+ KB


In [13]:
# # Replace all nan with median

cars_df['hp'] = pd.to_numeric(cars_df['hp'], errors='coerce')

# use lambda function to replace the median values in nan
cars_df = cars_df.apply(lambda x: x.fillna(x.median()), axis = 0)
# or use the below statement
# cars_df['hp'] = cars_df['hp'].fillna(cars_df['hp'].median())

print("Missing values in 'hp' column:", cars_df['hp'].isna().sum())


cars_df.head()



Missing values in 'hp' column: 0


,mpg,cyl,disp,hp,wt,acc,yr,car_type,origin_america,origin_asia,origin_europe
0,18.0,8,307.0,130.0,3504,12.0,70,0,1,0,0
1,15.0,8,350.0,165.0,3693,11.5,70,0,1,0,0
2,18.0,8,318.0,150.0,3436,11.0,70,0,1,0,0
3,16.0,8,304.0,150.0,3433,12.0,70,0,1,0,0
4,17.0,8,302.0,140.0,3449,10.5,70,0,1,0,0


In [14]:
cars_df.isna().sum()

mpg               0
cyl               0
disp              0
hp                0
wt                0
acc               0
yr                0
car_type          0
origin_america    0
origin_asia       0
origin_europe     0
dtype: int64

# Model building

In [17]:
X = cars_df.drop(['mpg'], axis=1)
y = cars_df[['mpg']]
print("X.column = ",X.columns)
print("y.column = ",y.columns)

X.column =  Index(['cyl', 'disp', 'hp', 'wt', 'acc', 'yr', 'car_type', 'origin_america',
       'origin_asia', 'origin_europe'],
      dtype='object')
y.column =  Index(['mpg'], dtype='object')


In [24]:
# scaling the data
X_s = preprocessing.scale(x)
X_s = pd.DataFrame(X_s, columns = X.columns) #converting scaled data into dataframe

y_s = preprocessing.scale(y)
y_s = pd.DataFrame(y_s,columns=y.columns)

X_s

,cyl,disp,hp,wt,acc,yr,car_type,origin_america,origin_asia,origin_europe
0,1.498191,1.090604,0.673118,0.630870,-1.295498,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
1,1.498191,1.503514,1.589958,0.854333,-1.477038,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
2,1.498191,1.196232,1.197027,0.550470,-1.658577,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
3,1.498191,1.061796,1.197027,0.546923,-1.295498,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
4,1.498191,1.042591,0.935072,0.565841,-1.840117,-1.627426,-1.062235,0.773559,-0.497643,-0.461968
...,...,...,...,...,...,...,...,...,...,...
393,-0.856321,-0.513026,-0.479482,-0.213324,0.011586,1.621983,0.941412,0.773559,-0.497643,-0.461968
394,-0.856321,-0.925936,-1.370127,-0.993671,3.279296,1.621983,0.941412,-1.292726,-0.497643,2.164651
395,-0.856321,-0.561039,-0.531873,-0.798585,-1.440730,1.621983,0.941412,0.773559,-0.497643,-0.461968
396,-0.856321,-0.705077,-0.662850,-0.408411,1.100822,1.621983,0.941412,0.773559,-0.497643,-0.461968


In [19]:
X_train, X_test, y_train, y_test = train_test_split(X_s,y_s,test_size=.30,random_state=0)
X_train.shape


(278, 10)

# Simple Linear Model

In [23]:
regression_model = LinearRegression()
regression_model.fit(X_train, y_train)


for idx,col_name in enumerate(X_train.columns):
    print('The coefficient for {} is {}'.format(col_name, regression_model.coef_[0][idx]))
    
intercept = regression_model.intercept_[0]
print('The intercept is {}'.format(intercept))

The coefficient for cyl is 0.24744479758946591
The coefficient for disp is 0.288382154460988
The coefficient for hp is -0.1899034268715287
The coefficient for wt is -0.6732229065111769
The coefficient for acc is 0.06754501540688149
The coefficient for yr is 0.34463640721172756
The coefficient for car_type is 0.31491491540037647
The coefficient for origin_america is -0.07682943694882888
The coefficient for origin_asia is 0.0633604889662
The coefficient for origin_europe is 0.03128335735147497
The intercept is -0.019500467624017425


# Regularized Ridge Regression

In [25]:
#alpha factor here is lambda (penalty term) which helps to reduce the magnitude of coeff

ridge_model = Ridge(alpha = 0.3)
ridge_model.fit(X_train, y_train)

print('Ridge model coef: {}'.format(ridge_model.coef_))
#As the data has 10 columns hence 10 coefficients appear here    

Ridge model coef: [ 0.24424435  0.27853222 -0.18980689 -0.66458446  0.06588077  0.34396213
  0.31169746 -0.07642734  0.06333336  0.03080065]


# Regularized Lasso Regression

In [27]:
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train,y_train)

print('Lasso model coef: {}'.format(lasso_model.coef_))
#As the data has 10 columns hence 10 coefficients appear here   

Lasso model coef: [-0.         -0.         -0.06203044 -0.48363379  0.          0.27163751
  0.09620861 -0.03490256  0.          0.        ]


# Score Comparison

In [ ]:
#Model score - r^2 or coeff of determinant
#r^2 = 1-(RSS/TSS) = Regression error/TSS 

#Simple Linear Model
print('Simple Linear Model\n')
print(regression_model.score(X_train, y_train))
print(regression_model.score(X_test, y_test))
print('*************************\n')


#Ridge
print("Ridge\n")
print(ridge_model.score(X_train, y_train))
print(ridge_model.score(X_test, y_test))
print('*************************\n')


#Lasso
print("Lasso\n")
print(lasso_model.score(X_train, y_train))
print(lasso_model.score(X_test, y_test))

Simple Linear Model

0.836163800114943
0.8439452810748137
*************************

Ridge

0.8361520170844985
0.8437853815947183
*************************

Lasso

0.7994535676270829
0.81026554865651
